# 🏥 Hospital Financial Performance & Cash Flow Analysis

## Project Goal

Analyze hospital financial data across:

- Patient payments
- Insurance reimbursements
- Government reimbursements
- Cash outflows
- Outstanding receivables
- Monthly financial performance

This notebook supports the Power BI project by providing a reproducible Python workflow for data understanding, validation, cleaning, and exploratory analysis.

> **Note:** The original source CSV/Excel files were not included with the extracted Power BI model.  
> Add the source datasets to the `data/` folder using the filenames shown below.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

DATA_DIR = Path("data")

expected_files = {
    "cash_outflows": "clean_cash_outflows.csv",
    "government": "clean_government.csv",
    "insurance": "clean_insurance.csv",
    "monthly_financial": "clean_monthly_financial.csv",
    "outstanding": "clean_outstanding.csv",
    "patient_payments": "clean_patient_payments.csv",
}

print("Expected source files:")
for key, filename in expected_files.items():
    print(f"- {filename}")


## 1. Load Available Datasets


In [ ]:
datasets = {}

for name, filename in expected_files.items():
    path = DATA_DIR / filename

    if path.exists():
        datasets[name] = pd.read_csv(path)
        print(f"Loaded {filename}: {datasets[name].shape}")
    else:
        print(f"Missing: {filename}")

if not datasets:
    print(
        "\nNo source CSV files were found. "
        "Add the exported source tables to the data/ folder and rerun this notebook."
    )


## 2. Data Understanding


In [ ]:
for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name.upper())
    print("=" * 80)
    print("Shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))
    print("\nData types:")
    display(df.dtypes.to_frame("dtype"))
    print("\nPreview:")
    display(df.head())


## 3. Data Quality Check


In [ ]:
quality_summary = []

for name, df in datasets.items():
    quality_summary.append({
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
    })

quality_df = pd.DataFrame(quality_summary)

if not quality_df.empty:
    display(quality_df)
else:
    print("Load source datasets to generate the quality summary.")


In [ ]:
for name, df in datasets.items():
    missing = (
        df.isna().sum()
          .to_frame("missing_count")
          .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
          .sort_values("missing_pct", ascending=False)
    )

    print(f"\nMissing-value profile: {name}")
    display(missing[missing["missing_count"] > 0].head(15))


## 4. Basic Cleaning


In [ ]:
cleaned = {}

for name, df in datasets.items():
    x = df.copy()

    # Remove exact duplicate rows
    x = x.drop_duplicates()

    # Trim text columns
    for col in x.select_dtypes(include="object").columns:
        x[col] = x[col].astype(str).str.strip().replace({"nan": np.nan})

    # Convert likely date columns
    date_cols = [c for c in x.columns if "date" in c.lower()]
    for col in date_cols:
        x[col] = pd.to_datetime(x[col], errors="coerce")

    cleaned[name] = x

    print(
        f"{name}: {df.shape} -> {x.shape} | "
        f"date columns detected: {date_cols}"
    )


## 5. Numeric Financial Summary


In [ ]:
for name, df in cleaned.items():
    numeric = df.select_dtypes(include=np.number)

    if numeric.empty:
        continue

    print(f"\nFinancial / numeric summary: {name}")
    display(numeric.describe().T)


## 6. Monthly Financial Trend Analysis


In [ ]:
monthly = cleaned.get("monthly_financial")

if monthly is not None:
    date_cols = [c for c in monthly.columns if "date" in c.lower() or "month" in c.lower()]
    numeric_cols = monthly.select_dtypes(include=np.number).columns.tolist()

    print("Potential date/month columns:", date_cols)
    print("Numeric columns:", numeric_cols)

    if date_cols and numeric_cols:
        date_col = date_cols[0]

        if not pd.api.types.is_datetime64_any_dtype(monthly[date_col]):
            monthly[date_col] = pd.to_datetime(monthly[date_col], errors="coerce")

        temp = monthly.dropna(subset=[date_col]).sort_values(date_col)

        plt.figure(figsize=(11, 5))

        for col in numeric_cols[:4]:
            plt.plot(temp[date_col], temp[col], marker="o", label=col)

        plt.title("Monthly Financial Trends")
        plt.xlabel("Period")
        plt.ylabel("Value")
        plt.legend()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    else:
        print("A usable date/month column and numeric financial columns are required.")
else:
    print("clean_monthly_financial.csv has not been added yet.")


## 7. Outstanding Receivables Analysis


In [ ]:
outstanding = cleaned.get("outstanding")

if outstanding is not None:
    numeric_cols = outstanding.select_dtypes(include=np.number).columns.tolist()

    print("Numeric columns available:")
    print(numeric_cols)

    if numeric_cols:
        display(
            outstanding[numeric_cols]
            .sum(numeric_only=True)
            .sort_values(ascending=False)
            .to_frame("total")
        )
else:
    print("clean_outstanding.csv has not been added yet.")


## 8. Patient Payment Analysis


In [ ]:
patient = cleaned.get("patient_payments")

if patient is not None:
    print("Patient-payment columns:")
    print(list(patient.columns))

    numeric_cols = patient.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = patient.select_dtypes(include="object").columns.tolist()

    if numeric_cols:
        display(patient[numeric_cols].describe().T)

    for col in categorical_cols[:3]:
        print(f"\nTop values - {col}")
        display(patient[col].value_counts().head(10).to_frame("count"))
else:
    print("clean_patient_payments.csv has not been added yet.")


## 9. Insurance & Government Reimbursements


In [ ]:
for dataset_name in ["insurance", "government"]:
    df = cleaned.get(dataset_name)

    if df is None:
        print(f"{dataset_name}: source file not added yet.")
        continue

    print("\n" + dataset_name.upper())
    print("Columns:", list(df.columns))

    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()

    if numeric_cols:
        totals = (
            df[numeric_cols]
            .sum(numeric_only=True)
            .sort_values(ascending=False)
        )
        display(totals.to_frame("total"))


## 10. Cash Outflow Analysis


In [ ]:
cash = cleaned.get("cash_outflows")

if cash is not None:
    numeric_cols = cash.select_dtypes(include=np.number).columns.tolist()
    categorical_cols = cash.select_dtypes(include="object").columns.tolist()

    if numeric_cols:
        display(cash[numeric_cols].describe().T)

    if categorical_cols and numeric_cols:
        category_col = categorical_cols[0]
        value_col = numeric_cols[0]

        summary = (
            cash.groupby(category_col, dropna=False)[value_col]
                .sum()
                .sort_values(ascending=False)
                .head(10)
        )

        display(summary.to_frame("total"))

        plt.figure(figsize=(9, 5))
        plt.barh(summary.sort_values().index.astype(str), summary.sort_values().values)
        plt.title("Top Cash Outflow Categories")
        plt.xlabel(value_col)
        plt.tight_layout()
        plt.show()
else:
    print("clean_cash_outflows.csv has not been added yet.")


## 11. Export Cleaned Data


In [ ]:
OUTPUT_DIR = Path("cleaned_data")
OUTPUT_DIR.mkdir(exist_ok=True)

for name, df in cleaned.items():
    output_path = OUTPUT_DIR / f"{name}_cleaned.csv"
    df.to_csv(output_path, index=False)
    print("Saved:", output_path)


## 12. Business Insight Checklist

After running the notebook with the real hospital datasets, summarize the actual results:

- Total cash inflows and outflows
- Revenue and expense trends over time
- Outstanding receivables
- Patient-payment collection patterns
- Insurance reimbursement performance
- Government reimbursement performance
- Major cash-outflow categories
- Areas with payment delays or financial gaps

Use only the values produced by the real datasets when writing the final conclusions.
